60 mins 1.2 secs

## Load libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm

## Config

In [2]:

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

spatial_cols = ["centroid_x", "centroid_y", "CoL_distance_km"]  # add others if you want, e.g. "CoL_distance_km"
# temporal signals: use STL lags + any macro time-varying series you want treated as "time"
temporal_macro_cols = ["base_rate", "GDP", "CPIH", "sdlt_perc_threshold"]  # optional; keep or extend
# socio-economic / structural (everything else you want *not* spatial, not temporal)
# (we'll build this as "remaining continuous cols" below)

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# GP tuning grid
gp_param_grid = {
    # space kernel (RBF)
    "gamma_space": [0.05, 0.1],
    "n_space": [200, 500],

    # time kernel (RBF on [t_month, cos, sin] + (optionally) temporal_macro_cols)
    "gamma_time": [0.05, 0.1],
    "n_time": [200, 500],

    # socio-economic kernel: choose either linear (no gamma) OR RBF-RFF
    # If you want it linear, set n_socio = 0 and we’ll just pass through raw socio features.
    "gamma_socio": [0.05, 0.1],
    "n_socio": [0, 300],  # 0 => linear / raw; >0 => RBF-RFF

    # weights for additive kernel
    "w_space": [1.0],
    "w_time": [1.0],
    "w_socio": [1.0],

    # ridge regularisation
    "alpha": [1e-3, 1e-2, 1e-1],
}


## Evaluation metric functions

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale


## Load data

In [4]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## Rolling STL feature builder

In [5]:
def add_rolling_stl_components(
    df: pd.DataFrame,
    entity_col: str,
    time_col: str,
    target_col: str,
    period: int = 12,
    min_history: int = 24,
    robust: bool = True,
    show_progress: bool = True,
) -> pd.DataFrame:
    """
    Time-safe rolling STL (one-sided).
    For each entity and each time t, fit STL on y[:t] and assign the last component values to time t.

    Outputs columns:
      - stl_trend
      - stl_seasonal
      - stl_resid

    Notes:
    - This is computationally heavier than "fit once on train then extrapolate".
    - It avoids leakage because STL at time t uses only <= t data.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    grouped = df.groupby(entity_col, sort=False)
    iterator = grouped if not show_progress else tqdm(grouped, desc="Rolling STL by LA", leave=False)

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        idx = sub.index.values

        # Rolling one-sided STL: start only when we have enough history
        for t in range(min_history - 1, len(y)):
            y_hist = y[: t + 1]

            # Skip if history contains NaNs
            if np.isnan(y_hist).any():
                continue

            try:
                stl = STL(y_hist, period=period, robust=robust)
                res = stl.fit()

                df.loc[idx[t], "stl_trend"] = float(res.trend[-1])
                df.loc[idx[t], "stl_seasonal"] = float(res.seasonal[-1])
                df.loc[idx[t], "stl_resid"] = float(res.resid[-1])

            except Exception:
                # If STL fails for numeric reasons at this t, leave NaNs
                continue

    return df

## Training with STL + Rolling CV

In [6]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================
from collections import defaultdict

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end   = dates[train_end_idx - 1]
    val_start   = dates[val_start_idx]
    val_end     = dates[val_end_idx - 1]

    fold_specs.append((train_start, train_end, val_start, val_end))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

# =========================================================
# MAIN LOOP: FOLDS → (lag_set → params)
# =========================================================
for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, start=1):
    print(f"\n=== Fold {fold_no}: "
          f"Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
          f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ===")

    # ---- slice fold train/val ----
    mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
    mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

    fold_train = df_tv.loc[mask_train].copy()
    fold_val   = df_tv.loc[mask_val].copy()

    fold_train["is_train"] = True
    fold_val["is_train"]   = False

    combined = pd.concat([fold_train, fold_val], axis=0).sort_values([ENTITY_COL, TIME_COL])

    combined = add_rolling_stl_components(
        combined,
        entity_col=ENTITY_COL,
        time_col=TIME_COL,
        target_col=TARGET_COL,
        period=12,
        min_history=24,
        robust=True,
        show_progress=True,
    )
    combined = combined.sort_values([ENTITY_COL, TIME_COL])

    fold_origin = combined[TIME_COL].min()
    combined["t_month"] = (
        (combined[TIME_COL].dt.year - fold_origin.year) * 12
        + (combined[TIME_COL].dt.month - fold_origin.month)
    ).astype(float)

    # seasonal month-of-year embedding for a periodic kernel via RBF on a circle
    combined["month_num"] = combined[TIME_COL].dt.month.astype(float)
    combined["month_cos"] = np.cos(2 * np.pi * combined["month_num"] / 12.0)
    combined["month_sin"] = np.sin(2 * np.pi * combined["month_num"] / 12.0)

    time_cols = ["t_month", "month_cos", "month_sin"]

    # create lag columns for ALL lags (superset) on STL components
    for lag in all_lags:
        for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
            col = f"{comp}_lag{lag}"
            combined[col] = combined.groupby(ENTITY_COL)[comp].shift(lag)

    # ---------------------------------------------------------
    # For this fold: loop over lag_set, then RF params
    # Using the precomputed lag columns in `combined`
    # ---------------------------------------------------------
    for lag_set in lag_combinations:
        print(f"  Lag set: {lag_set}")
        lag_cols = [
            f"{comp}_lag{lag}"
            for comp in ["stl_trend", "stl_seasonal", "stl_resid"]
            for lag in lag_set
        ]

        temporal_cols = lag_cols + [c for c in temporal_macro_cols if c in continuous_cols]

        # socio-economic = continuous minus (spatial + temporal_macro) plus categorical (handled separately)
        socio_cont_cols = [c for c in continuous_cols if c not in set(spatial_cols + temporal_macro_cols)]

        # split back into train/val
        fold_train_lag = combined[combined["is_train"]].copy()
        fold_val_lag   = combined[~combined["is_train"]].copy()

        # require ALL FEATURES (continuous + categorical + lags) present
        full_feature_cols = continuous_cols + categorical_cols + lag_cols
        fold_train_lag = fold_train_lag.dropna(subset=full_feature_cols)
        fold_val_lag   = fold_val_lag.dropna(subset=full_feature_cols)

        if fold_train_lag.empty or fold_val_lag.empty:
            print("    (skip: no data after feature drop)")
            continue

        # -------------------------------------------------
        # BUILD X BLOCKS
        # -------------------------------------------------
        feature_cols_all = (
            spatial_cols
            + time_cols
            + temporal_cols
            + socio_cont_cols
            + categorical_cols
        )
        fold_train_lag = fold_train_lag.dropna(subset=feature_cols_all)
        fold_val_lag   = fold_val_lag.dropna(subset=feature_cols_all)

        if fold_train_lag.empty or fold_val_lag.empty:
            print("    (skip: no data after feature drop)")
            continue

        # blocks
        X_space_train = fold_train_lag[spatial_cols].copy()
        X_space_val   = fold_val_lag[spatial_cols].copy()

        # time block can include explicit time cols + (optional) macro temporal cols + STL lag cols
        X_time_train = fold_train_lag[time_cols + temporal_cols].copy()
        X_time_val   = fold_val_lag[time_cols + temporal_cols].copy()

        # socio block: structural + socio-economic continuous + categoricals
        X_socio_train = fold_train_lag[socio_cont_cols + categorical_cols].copy()
        X_socio_val   = fold_val_lag[socio_cont_cols + categorical_cols].copy()

        y_train = fold_train_lag[TARGET_COL].values
        y_val   = fold_val_lag[TARGET_COL].values

        # -------------------------------------------------
        # SCALE BLOCKS (train-only fit)
        # -------------------------------------------------
        sc_space = StandardScaler()
        X_space_train.loc[:, spatial_cols] = sc_space.fit_transform(X_space_train[spatial_cols])
        X_space_val.loc[:, spatial_cols]   = sc_space.transform(X_space_val[spatial_cols])

        sc_time = StandardScaler()
        time_block_cols = time_cols + temporal_cols

        # ensure float dtype before scaling
        X_time_train[time_block_cols] = X_time_train[time_block_cols].astype(float)
        X_time_val[time_block_cols]   = X_time_val[time_block_cols].astype(float)

        X_time_train.loc[:, time_block_cols] = sc_time.fit_transform(X_time_train[time_block_cols])
        X_time_val.loc[:, time_block_cols]   = sc_time.transform(X_time_val[time_block_cols])
        
        sc_socio = StandardScaler()
        # scale only continuous socio columns (categoricals already 0/1)
        socio_scale_cols = socio_cont_cols
        X_socio_train.loc[:, socio_scale_cols] = sc_socio.fit_transform(X_socio_train[socio_scale_cols])
        X_socio_val.loc[:, socio_scale_cols]   = sc_socio.transform(X_socio_val[socio_scale_cols])

        # -------------------------------------------------
        # SCALE y 
        # -------------------------------------------------
        y_scaler = StandardScaler()
        y_train_raw = y_train.reshape(-1, 1)
        y_val_raw   = y_val.reshape(-1, 1)

        y_train_scaled = y_scaler.fit_transform(y_train_raw).ravel()
        y_val_scaled   = y_scaler.transform(y_val_raw).ravel()

        # inner loop over GP hyperparameters
        for params in ParameterGrid(gp_param_grid):
            lag_key = tuple(lag_set)
            params_key = tuple(sorted(params.items()))
            key = (lag_key, params_key)

            if key not in metrics_store:
                metrics_store[key] = {
                    "lag_set": lag_key,
                    "params": params,
                    "mae": [],
                    "rmse": [],
                    "smape": [],
                    "mase": [],
                    "folds": 0,
                }

            # -----------------------------
            # Space kernel: RBF-RFF
            # -----------------------------
            rff_space = RBFSampler(
                gamma=params["gamma_space"],
                n_components=params["n_space"],
                random_state=42,
            )
            Zs_tr = rff_space.fit_transform(X_space_train)
            Zs_va = rff_space.transform(X_space_val)

            # -----------------------------
            # Time kernel: RBF-RFF
            # (because month_cos/sin are included, this behaves like a periodic kernel component)
            # -----------------------------
            rff_time = RBFSampler(
                gamma=params["gamma_time"],
                n_components=params["n_time"],
                random_state=43,
            )
            Zt_tr = rff_time.fit_transform(X_time_train)
            Zt_va = rff_time.transform(X_time_val)

            # -----------------------------
            # Socio-economic kernel:
            #  - if n_socio == 0 : linear kernel (raw features)
            #  - else            : RBF-RFF
            # -----------------------------
            if params["n_socio"] == 0:
                Ze_tr = X_socio_train.values
                Ze_va = X_socio_val.values
            else:
                rff_socio = RBFSampler(
                    gamma=params["gamma_socio"],
                    n_components=params["n_socio"],
                    random_state=44,
                )
                Ze_tr = rff_socio.fit_transform(X_socio_train)
                Ze_va = rff_socio.transform(X_socio_val)

            # -----------------------------
            # Additive kernel via concatenation
            # scale each block by sqrt(weight)
            # -----------------------------
            ws = np.sqrt(params["w_space"])
            wt = np.sqrt(params["w_time"])
            we = np.sqrt(params["w_socio"])

            Z_train = np.hstack([ws * Zs_tr, wt * Zt_tr, we * Ze_tr])
            Z_val   = np.hstack([ws * Zs_va, wt * Zt_va, we * Ze_va])

            gp_ridge = Ridge(alpha=params["alpha"])
            gp_ridge.fit(Z_train, y_train_scaled)

            y_pred_scaled = gp_ridge.predict(Z_val)
            y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

            y_val_true = y_val_raw.ravel()
            y_train_true = y_train_raw.ravel()

            metrics_store[key]["mae"].append(mae(y_val_true, y_pred))
            metrics_store[key]["rmse"].append(rmse(y_val_true, y_pred))
            metrics_store[key]["smape"].append(smape(y_val_true, y_pred))
            metrics_store[key]["mase"].append(mase(y_val_true, y_pred, y_train_true))
            metrics_store[key]["folds"] += 1





Number of folds: 5

=== Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03 ===


  Lag set: [1, 12]
  Lag set: [1, 2, 12]
  Lag set: [1, 2, 3, 12]
  Lag set: [1, 2, 3, 4, 5, 6, 12]
  Lag set: [1, 12, 24]
  Lag set: [1, 2, 12, 24]
  Lag set: [1, 2, 3, 12, 24]
  Lag set: [1, 2, 3, 4, 5, 6, 12, 24]

=== Fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03 ===


  Lag set: [1, 12]
  Lag set: [1, 2, 12]
  Lag set: [1, 2, 3, 12]
  Lag set: [1, 2, 3, 4, 5, 6, 12]
  Lag set: [1, 12, 24]
  Lag set: [1, 2, 12, 24]
  Lag set: [1, 2, 3, 12, 24]
  Lag set: [1, 2, 3, 4, 5, 6, 12, 24]

=== Fold 3: Train 2009-04–2019-03, Val 2019-04–2020-03 ===


  Lag set: [1, 12]
  Lag set: [1, 2, 12]
  Lag set: [1, 2, 3, 12]
  Lag set: [1, 2, 3, 4, 5, 6, 12]
  Lag set: [1, 12, 24]
  Lag set: [1, 2, 12, 24]
  Lag set: [1, 2, 3, 12, 24]
  Lag set: [1, 2, 3, 4, 5, 6, 12, 24]

=== Fold 4: Train 2010-04–2020-03, Val 2020-04–2021-03 ===


  Lag set: [1, 12]
  Lag set: [1, 2, 12]
  Lag set: [1, 2, 3, 12]
  Lag set: [1, 2, 3, 4, 5, 6, 12]
  Lag set: [1, 12, 24]
  Lag set: [1, 2, 12, 24]
  Lag set: [1, 2, 3, 12, 24]
  Lag set: [1, 2, 3, 4, 5, 6, 12, 24]

=== Fold 5: Train 2011-04–2021-03, Val 2021-04–2022-03 ===


  Lag set: [1, 12]
  Lag set: [1, 2, 12]
  Lag set: [1, 2, 3, 12]
  Lag set: [1, 2, 3, 4, 5, 6, 12]
  Lag set: [1, 12, 24]
  Lag set: [1, 2, 12, 24]
  Lag set: [1, 2, 3, 12, 24]
  Lag set: [1, 2, 3, 4, 5, 6, 12, 24]


## Results

In [7]:
# =========================================================
# AGGREGATE RESULTS OVER FOLDS
# =========================================================
rows = []
for key, val in metrics_store.items():
    if val["folds"] == 0:
        continue
    rows.append({
        "model_type": "SparseGP_RFF",
        "lag_set": val["lag_set"],
        "params": val["params"],
        "folds": val["folds"],
        "MAE_mean":   float(np.mean(val["mae"])),
        "MAE_std":    float(np.std(val["mae"])),
        "RMSE_mean":  float(np.mean(val["rmse"])),
        "RMSE_std":   float(np.std(val["rmse"])),
        "sMAPE_mean": float(np.mean(val["smape"])),
        "sMAPE_std":  float(np.std(val["smape"])),
        "MASE_mean":  float(np.mean(val["mase"])),
        "MASE_std":   float(np.std(val["mase"])),
    })

results_df = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("../../results/sparsegp_leakfree_stl_rollingcv_results.csv", index=False)

      model_type        lag_set  \
0   SparseGP_RFF  (1, 2, 3, 12)   
1   SparseGP_RFF  (1, 2, 3, 12)   
2   SparseGP_RFF  (1, 2, 3, 12)   
3   SparseGP_RFF  (1, 2, 3, 12)   
4   SparseGP_RFF  (1, 2, 3, 12)   
5   SparseGP_RFF  (1, 2, 3, 12)   
6   SparseGP_RFF  (1, 2, 3, 12)   
7   SparseGP_RFF  (1, 2, 3, 12)   
8   SparseGP_RFF  (1, 2, 3, 12)   
9   SparseGP_RFF  (1, 2, 3, 12)   
10  SparseGP_RFF  (1, 2, 3, 12)   
11  SparseGP_RFF  (1, 2, 3, 12)   
12  SparseGP_RFF  (1, 2, 3, 12)   
13  SparseGP_RFF  (1, 2, 3, 12)   
14  SparseGP_RFF  (1, 2, 3, 12)   
15  SparseGP_RFF  (1, 2, 3, 12)   
16  SparseGP_RFF  (1, 2, 3, 12)   
17  SparseGP_RFF  (1, 2, 3, 12)   
18  SparseGP_RFF  (1, 2, 3, 12)   
19  SparseGP_RFF  (1, 2, 3, 12)   

                                               params  folds      MAE_mean  \
0   {'alpha': 0.1, 'gamma_socio': 0.1, 'gamma_spac...      5  13535.436621   
1   {'alpha': 0.1, 'gamma_socio': 0.05, 'gamma_spa...      5  13535.436621   
2   {'alpha': 0.1, 'gamma_soci

In [8]:
print(results_df.head(20))

      model_type        lag_set  \
0   SparseGP_RFF  (1, 2, 3, 12)   
1   SparseGP_RFF  (1, 2, 3, 12)   
2   SparseGP_RFF  (1, 2, 3, 12)   
3   SparseGP_RFF  (1, 2, 3, 12)   
4   SparseGP_RFF  (1, 2, 3, 12)   
5   SparseGP_RFF  (1, 2, 3, 12)   
6   SparseGP_RFF  (1, 2, 3, 12)   
7   SparseGP_RFF  (1, 2, 3, 12)   
8   SparseGP_RFF  (1, 2, 3, 12)   
9   SparseGP_RFF  (1, 2, 3, 12)   
10  SparseGP_RFF  (1, 2, 3, 12)   
11  SparseGP_RFF  (1, 2, 3, 12)   
12  SparseGP_RFF  (1, 2, 3, 12)   
13  SparseGP_RFF  (1, 2, 3, 12)   
14  SparseGP_RFF  (1, 2, 3, 12)   
15  SparseGP_RFF  (1, 2, 3, 12)   
16  SparseGP_RFF  (1, 2, 3, 12)   
17  SparseGP_RFF  (1, 2, 3, 12)   
18  SparseGP_RFF  (1, 2, 3, 12)   
19  SparseGP_RFF  (1, 2, 3, 12)   

                                               params  folds      MAE_mean  \
0   {'alpha': 0.1, 'gamma_socio': 0.1, 'gamma_spac...      5  13535.436621   
1   {'alpha': 0.1, 'gamma_socio': 0.05, 'gamma_spa...      5  13535.436621   
2   {'alpha': 0.1, 'gamma_soci